# Notebook Evals Dashboard

This notebook acts as a lightweight dashboard for experiment evaluation.

The goal is to give us a browser-like analysis flow inside Jupyter before we build a local app. It focuses on three questions:

- How accurate is the agent overall?
- Is it selecting the right estimator for the data available?
- Which concrete runs explain the aggregate failures?


## Step 1: Configure The Dashboard Data Source

This cell lets us either run a fresh experiment or reuse a saved experiment output file.

Why this matters:

- a fresh run is useful while iterating on prompts or graph behavior
- a saved JSON file is useful when we want stable dashboard views for discussion and comparison


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from iroas_agent.dashboard import (
    EXPERIMENT_OUTPUTS_DIR,
    format_trajectory,
    load_or_create_experiment_output,
    metric_cards,
    results_frame,
    selection_matrix,
    slice_summary,
    tool_usage,
    worst_runs,
)
from iroas_agent.runner import run_experiment


## Configure The Artifact Workflow

This notebook is designed to be fast to reopen. Instead of rerunning the agent every time, we point it at a saved experiment artifact in `data/experiment_outputs/`.

Use `refresh_experiment = False` when you want to inspect a previously saved run. Flip it to `True` only when you intentionally want to regenerate the artifact by rerunning the agent.


In [ ]:
experiment_path = EXPERIMENT_OUTPUTS_DIR / "latest.json"
dataset_path = 'data/synthetic_campaigns.csv'
sample_size = 20
refresh_experiment = False

{
    'experiment_path': str(experiment_path),
    'artifact_exists_now': experiment_path.exists(),
    'refresh_experiment': refresh_experiment,
    'dataset_path': dataset_path,
}


## Load Existing Results Or Create Them Once

This cell is the key quality-of-life improvement for the dashboard. It tries to load a saved experiment artifact first. If no artifact exists yet, or if `refresh_experiment = True`, it runs the experiment once and writes the JSON artifact to disk for future dashboard sessions.


In [ ]:
artifact_existed_before = experiment_path.exists()

output = load_or_create_experiment_output(
    experiment_path,
    lambda: run_experiment(dataset_path=dataset_path, sample_size=sample_size),
    refresh=refresh_experiment,
)

{
    'artifact_existed_before': artifact_existed_before,
    'artifact_exists_after': experiment_path.exists(),
    'using_saved_artifact': artifact_existed_before and not refresh_experiment,
    'experiment_path': str(experiment_path),
    'result_keys': list(output.keys()),
}


In [ ]:
results = output['results']
frame = results_frame(results)
frame.head()

## Overview Cards

This section is the high-level health check.

The metrics are intentionally small in number so they are easy to scan:

- number of runs
- MSE
- bias
- mean absolute error
- average number of steps
- preferred-estimator rate

The preferred-estimator rate is especially useful for agent analysis because it tells us whether the policy is selecting the strongest available causal estimator most of the time.

In [ ]:
cards = metric_cards(frame)
pd.Series(cards)

## Availability Breakdown

This section slices quality by the type of evidence available.

Why this matters:

- the agent should not behave the same way in `lift_only` and `neither`
- a good policy should look strongest when high-quality causal evidence is present
- if one slice is much worse than the others, that usually points to a clear design problem


In [ ]:
slice_summary(frame, 'availability_type')

## Estimator Selection Dashboard

This section focuses on the agent's policy rather than just final numeric accuracy.

The matrix answers a direct question: given the available evidence, which estimator did the agent end up trusting?

This is one of the most important panels in the whole dashboard because it helps separate:

- estimator weakness
- policy weakness


In [ ]:
selection_matrix(frame)

In [ ]:
tool_usage(frame)

## Error Dashboard

This section visualizes how predictions differ from the hidden truth.

What the charts show:

- a scatterplot of predicted iROAS vs true iROAS
- a histogram of residuals

Why this matters:

- the scatterplot shows calibration and extreme misses
- the residual histogram shows whether the system tends to overestimate or underestimate


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(frame['true_iROAS'], frame['predicted_iROAS'], alpha=0.65)
lo = min(frame['true_iROAS'].min(), frame['predicted_iROAS'].min())
hi = max(frame['true_iROAS'].max(), frame['predicted_iROAS'].max())
axes[0].plot([lo, hi], [lo, hi], linestyle='--')
axes[0].set_title('Predicted vs True iROAS')
axes[0].set_xlabel('True iROAS')
axes[0].set_ylabel('Predicted iROAS')

axes[1].hist(frame['error'], bins=20)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Prediction Error')
axes[1].set_ylabel('Count')
plt.tight_layout()

## Behavior Dashboard

This section looks at the ReAct policy itself.

Useful questions here are:

- how often does the agent start with diagnostics
- how often does it use more than one estimator
- what action sequences are most common
- does extra reasoning correlate with better or worse outcomes


In [ ]:
behavior_summary = pd.Series({
    'diagnostics_first_rate': (frame['first_action'] == 'diagnostics_tool').mean(),
    'multi_estimator_rate': frame['used_multiple_estimators'].mean(),
    'avg_step_count': frame['step_count'].mean(),
})
behavior_summary

In [ ]:
frame['tool_sequence'].value_counts().head(10)

## Failure Explorer

This is the bridge from aggregate metrics to concrete debugging.

The table below ranks the largest mistakes. That is usually the fastest path to improving the agent because the worst runs tend to reveal:

- bad estimator selection
- premature stopping
- over-trusting weak evidence
- underweighting reliability signals


In [ ]:
worst = worst_runs(frame, top_n=10)
worst

## Run Detail Drill-Down

This cell lets us inspect one concrete run in detail.

To use it:

- set `selected_campaign_id` to a row from the failure table
- inspect the selected run's metadata, explanation, and trajectory

This is the notebook equivalent of clicking into a run detail drawer in a future app dashboard.

In [ ]:
selected_campaign_id = worst.iloc[0]['campaign_id'] if not worst.empty else frame.iloc[0]['campaign_id']
selected_run = next(row for row in results if row['campaign_id'] == selected_campaign_id)

{
    'campaign_id': selected_run['campaign_id'],
    'prediction': selected_run['prediction'],
    'metadata': selected_run['metadata'],
    'evidence_log': selected_run.get('evidence_log', []),
}

In [ ]:
print(format_trajectory(selected_run))

## What This Dashboard Gives Us

This notebook dashboard is intentionally a bridge artifact.

It is more structured than an ad hoc notebook, but less engineered than a local app. That makes it a good place to answer questions like:

- Which metrics actually matter most?
- Which filters do we use repeatedly?
- Which drill-downs are essential?
- What should the future app dashboard prioritize?

Once this notebook feels useful, we can convert its strongest sections into a local browser app with much less guesswork.